# 01 — Télécharger les prix (Yahoo Finance / FRED)

Ce notebook télécharge les prix actions/indices/FX (Yahoo) et taux (FRED, optionnel) et crée un **`prices.csv`** au format `timestamp,asset,price`.

In [ ]:
# Installer les dépendances (exécuter une seule fois par environnement)
%pip install -q yfinance pandas numpy fredapi

: 

In [ ]:
import os, pandas as pd, numpy as np
import yfinance as yf
from datetime import datetime

# --- Paramètres à adapter ---
START = "1997-02-07"
END   = datetime.utcnow().strftime("%Y-%m-%d")  # aujourd'hui
INTERVAL = "1d"  # ou "1h" pour intraday

# Mapping label -> ticker Yahoo
STOCKS  = {"AAPL":"AAPL", "MSFT":"MSFT"}
INDICES = {"SPX":"^GSPC", "CAC40":"^FCHI"}
FX      = {"EURUSD":"EURUSD=X", "USDJPY":"USDJPY=X"}

"SX5E": px_sx5e[['bucket','price']],

OUT_CSV = "prices.csv"  # sortie unifiée


C:\Users\Garance Latieule\AppData\Local\Temp\ipykernel_77792\1754697290.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  END   = datetime.utcnow().strftime("%Y-%m-%d")  # aujourd'hui


In [ ]:
def fetch_yf(map_label2tick, start, end, interval):
    if not map_label2tick:
        return pd.DataFrame(columns=["timestamp","asset","price"])
    labels_by_ticker = {v:k for k,v in map_label2tick.items()}
    data = yf.download(list(map_label2tick.values()), start=start, end=end, interval=interval, auto_adjust=False, progress=False)
    if isinstance(data.columns, pd.MultiIndex):
        close = data["Close"]
    else:
        close = data["Close"] if "Close" in data.columns else data
    if isinstance(close, pd.Series):
        close = close.to_frame()
    close = close.rename(columns=lambda c: labels_by_ticker.get(c, c))
    out = close.stack().reset_index()
    out.columns = ["timestamp","asset","price"]
    return out.dropna(subset=["price"])

In [ ]:
# Téléchargement Yahoo
frames = []
for mapping in (STOCKS, INDICES, FX):
    df = fetch_yf(mapping, START, END, INTERVAL)
    frames.append(df)
prices = pd.concat(frames, ignore_index=True)
prices = prices.sort_values(["asset","timestamp"]).reset_index(drop=True)
prices.to_csv(OUT_CSV, index=False)
print(f"Sauvé {len(prices):,} lignes dans {OUT_CSV}")
prices.head()

Sauvé 42,172 lignes dans prices.csv


,timestamp,asset,price
0,1997-02-07,AAPL,0.141183
1,1997-02-10,AAPL,0.139509
2,1997-02-11,AAPL,0.140067
3,1997-02-12,AAPL,0.140625
4,1997-02-13,AAPL,0.143973


### (Optionnel) Ajouter des taux FRED
Créez une clé API gratuite sur FRED, puis :

```bash
export FRED_API_KEY=VOTRE_CLE
```


In [ ]:
# Optionnel : ajouter des séries FRED
USE_FRED = False  # passez à True si vous voulez ajouter des taux
FRED_SERIES = ["DGS10","DGS2"]

if USE_FRED:
    try:
        from fredapi import Fred
        fred = Fred(api_key=os.environ.get("FRED_API_KEY",""))
        fred_frames = []
        for sid in FRED_SERIES:
            s = fred.get_series(sid, observation_start=START, observation_end=END)
            df = s.to_frame(name="price")
            df["timestamp"] = df.index.tz_localize(None)
            df["asset"] = sid
            fred_frames.append(df[["timestamp","asset","price"]])
        if fred_frames:
            fred_df = pd.concat(fred_frames, ignore_index=True)
            prices = pd.concat([prices, fred_df], ignore_index=True)
            prices = prices.sort_values(["asset","timestamp"]).reset_index(drop=True)
            prices.to_csv(OUT_CSV, index=False)
            print(f"Ajouté FRED: total {len(prices):,} lignes → {OUT_CSV}")
    except Exception as e:
        print("FRED indisponible:", e)

In [3]:
import os
import pandas as pd
import yfinance as yf
from datetime import datetime, timezone

# ============ Paramètres ============
START = "1997-02-07"
END   = datetime.now(timezone.utc).strftime("%Y-%m-%d")  # aujourd'hui (UTC aware)
INTERVAL = "1d"
OUT_CSV = "prices.csv"

# ============ Mappings ============
# Indices / Proxies (Yahoo n’a pas toujours l’index cash)
INDICES = {
    "SX5E": "EXW1.DE",   # iShares EURO STOXX 50 UCITS ETF (proxy index ^STOXX50E)
    "SX7E": "EXV1.DE",   # iShares STOXX Europe 600 Banks UCITS ETF (proxy Banks)
}

# Futures EUREX: souvent indisponibles sur Yahoo (FGBL=F / FGBS=F => 404 chez toi)
# On les laisse vides par défaut. Si tu veux retenter: {"FGBL":"FGBL=F","FGBS":"FGBS=F"}
FUTURES = {}

# FX
FX = {
    "EURUSD": "EURUSD=X",
}

# (Optionnel) Actions
STOCKS = {
    # "AAPL": "AAPL",
    # "MSFT": "MSFT",
}

# ============ Téléchargement robuste par ticker ============
def fetch_one(label: str, ticker: str, start: str, end: str, interval: str) -> pd.DataFrame:
    """Télécharge un ticker Yahoo -> DataFrame long [timestamp, asset, price].
       Retourne df vide si échec / pas de données."""
    try:
        df = yf.download(ticker, start=start, end=end, interval=interval,
                         auto_adjust=False, progress=False, threads=True)
        if df is None or df.empty:
            print(f"[WARN] Pas de données pour {label} ({ticker})")
            return pd.DataFrame(columns=["timestamp","asset","price"])
        # Close ou unique colonne
        close = df["Close"] if "Close" in df.columns else df.squeeze()
        if isinstance(close, pd.Series):
            close = close.to_frame(name=label)
        else:
            close = close.rename(columns={ticker: label})
        out = close.reset_index().rename(columns={"Date":"timestamp"})
        out = out[["timestamp", label]].rename(columns={label:"price"})
        out["asset"] = label
        # normalise timestamp
        out["timestamp"] = pd.to_datetime(out["timestamp"], errors="coerce").dt.tz_localize(None)
        out = out.dropna(subset=["price"])
        return out[["timestamp","asset","price"]]
    except Exception as e:
        print(f"[ERROR] {label} ({ticker}) : {e}")
        return pd.DataFrame(columns=["timestamp","asset","price"])

def fetch_map(label2ticker: dict) -> pd.DataFrame:
    frames = []
    for label, tick in label2ticker.items():
        frames.append(fetch_one(label, tick, START, END, INTERVAL))
    # filtre les vides pour éviter FutureWarning
    frames = [f for f in frames if f is not None and not f.empty]
    if not frames:
        return pd.DataFrame(columns=["timestamp","asset","price"])
    return pd.concat(frames, ignore_index=True)

# ============ Run ============ 
frames = []
for mapping in (INDICES, FUTURES, FX, STOCKS):
    df = fetch_map(mapping)
    if not df.empty:
        frames.append(df)

if frames:
    prices = pd.concat(frames, ignore_index=True)
    prices = prices.sort_values(["asset","timestamp"]).reset_index(drop=True)
else:
    prices = pd.DataFrame(columns=["timestamp","asset","price"])

prices.to_csv(OUT_CSV, index=False)
print(f"Sauvé {len(prices):,} lignes dans {OUT_CSV}")
print(prices.head())


Sauvé 14,753 lignes dans prices.csv
Ticker  timestamp   asset     price
0      2003-12-01  EURUSD  1.196501
1      2003-12-02  EURUSD  1.208897
2      2003-12-03  EURUSD  1.212298
3      2003-12-04  EURUSD  1.208094
4      2003-12-05  EURUSD  1.218695
